# Transformers: XLM-R and mBERT (Methods Table 1, TODO 1)

Fine-tunes `FacebookAI/xlm-roberta-base` (first-token representation) and
`google-bert/bert-base-multilingual-cased` (pooled CLS) with one multilabel head
(three sigmoid outputs, BCE loss) on `data/05_final/final_dataset.csv`.

Protocol is identical to `src/train_classical.py`: 5-fold grouped CV, for test fold k the
validation fold is (k+1) mod 5 and the other three folds are training. Early stopping and the
per-label decision threshold (`metrics.best_threshold`) use the validation fold only and are
frozen before the test fold is scored. Seeds 13, 21, 42, 87, 100 → 25 runs per system.
E3: mBERT with lower-cased input (`mBERT-lower`).

Runs locally on the RTX 3070 Ti with `..\.venv`. Every finished run is cached in
`results/transformers/runs/`, so re-running the notebook skips completed runs.
Outputs contain ids, labels and scores only — no comment text is printed or saved.

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # needed for deterministic cuBLAS

import gc
import json
import math
import platform
import random
import subprocess
import sys
import time
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import sklearn
import tokenizers
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import snapshot_download
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from metrics import LABELS, LANGS, best_threshold, by_language, evaluate

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
assert torch.cuda.is_available(), "CUDA GPU not available"
DEVICE = "cuda"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip()
ENV = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "cudnn": torch.backends.cudnn.version(), "transformers": transformers.__version__,
       "tokenizers": tokenizers.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "scikit_learn": sklearn.__version__, "gpu": torch.cuda.get_device_name(0),
       "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
       "nvidia_driver": driver, "os": platform.platform()}
for k, v in ENV.items():
    print(f"{k:14s} {v}")

C:\Users\yerlan\Documents\uni\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python         3.12.10
torch          2.14.1+cu130
cuda           13.0
cudnn          92400
transformers   5.20.2
tokenizers     0.23.3
numpy          2.5.3
pandas         3.0.6
scikit_learn   1.9.1
gpu            NVIDIA GeForce RTX 3070 Ti
gpu_memory_gb  8.0
nvidia_driver  610.47
os             Windows-11-10.0.26200-SP0


In [2]:
SMOKE = False   # True: 1 seed, 1 fold, 1 epoch, written to results/transformers_smoke/

SEEDS = [13, 21, 42, 87, 100]
FOLDS = list(range(5))
N_FOLDS = 5
MAX_LEN = 128
LR = 2e-5
WEIGHT_DECAY = 0.01
BATCH = 16           # micro-batch; x ACCUM = effective batch 32
ACCUM = 2
WARMUP = 0.10        # share of the maximum number of optimizer steps
MAX_EPOCHS = 10
PATIENCE = 2         # epochs without validation-loss improvement
DROPOUT = 0.1
AMP_DTYPE = torch.bfloat16

SYSTEMS = {
    "XLM-R": {"model": "FacebookAI/xlm-roberta-base", "pooling": "first", "lower": False},
    "mBERT": {"model": "google-bert/bert-base-multilingual-cased", "pooling": "pooler", "lower": False},
    "mBERT-lower": {"model": "google-bert/bert-base-multilingual-cased", "pooling": "pooler", "lower": True},
}

if SMOKE:
    SEEDS, FOLDS, MAX_EPOCHS = [42], [0], 1
OUT = Path("results/transformers_smoke" if SMOKE else "results/transformers")
RUNS = OUT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print(f"{len(SYSTEMS)} systems x {len(SEEDS)} seeds x {len(FOLDS)} folds = "
      f"{len(SYSTEMS) * len(SEEDS) * len(FOLDS)} runs -> {OUT}/")

3 systems x 5 seeds x 5 folds = 75 runs -> results\transformers/


In [3]:
df = pd.read_csv("data/05_final/final_dataset.csv")
assert df["id"].is_unique and sorted(df["fold"].unique()) == list(range(N_FOLDS))
print(f"{len(df)} comments")
print(df.groupby("lang")[LABELS].agg(["size", "sum"]).to_string())


def prep(texts, spec):
    texts = texts.fillna("").astype(str)
    return texts.str.lower() if spec["lower"] else texts


TOKENIZERS = {name: AutoTokenizer.from_pretrained(spec["model"]) for name, spec in SYSTEMS.items()}
REVISIONS = {spec["model"]: Path(snapshot_download(spec["model"], local_files_only=True)).name
             for spec in SYSTEMS.values()}
print(REVISIONS)

285 comments
     toxic     offensive     hate    
      size sum      size sum size sum
lang                                 
CS      44  18        44  10   44   2
KK     124  33       124  23  124   5
RU     117  43       117  34  117  11


{'FacebookAI/xlm-roberta-base': 'e73636d4f797dec63c3081bb6ed5c7b0bb3f2089', 'google-bert/bert-base-multilingual-cased': '3f076fdb1ab68d5b2880cb87a0886f315b8146f8'}


## Truncation rate (> 128 tokens including special tokens)

In [4]:
rows = []
for name, spec in SYSTEMS.items():
    ids = TOKENIZERS[name](list(prep(df["text"], spec)), add_special_tokens=True,
                           truncation=False)["input_ids"]
    d = df[["lang"]].assign(n_tokens=[len(x) for x in ids])
    for lang in LANGS + ["All"]:
        s = d if lang == "All" else d[d["lang"] == lang]
        n_trunc = int((s["n_tokens"] > MAX_LEN).sum())
        rows.append({"system": name, "lang": lang, "n": len(s), "truncated": n_trunc,
                     "rate": n_trunc / len(s), "median_tokens": float(s["n_tokens"].median()),
                     "max_tokens": int(s["n_tokens"].max())})
truncation = pd.DataFrame(rows)
truncation.to_csv(OUT / "truncation.csv", index=False)
print(truncation.to_string(index=False))

     system lang   n  truncated     rate  median_tokens  max_tokens
      XLM-R   KK 124          6 0.048387           23.0         193
      XLM-R   RU 117          7 0.059829           28.0         850
      XLM-R   CS  44          1 0.022727           33.0         148
      XLM-R  All 285         14 0.049123           26.0         850
      mBERT   KK 124         13 0.104839           27.0         255
      mBERT   RU 117          8 0.068376           32.0        1021
      mBERT   CS  44          1 0.022727           35.5         162
      mBERT  All 285         22 0.077193           31.0        1021
mBERT-lower   KK 124         10 0.080645           27.0         251
mBERT-lower   RU 117          8 0.068376           31.0        1005
mBERT-lower   CS  44          1 0.022727           35.5         161
mBERT-lower  All 285         19 0.066667           29.0        1005


## Model and training

In [5]:
class MultiLabelModel(torch.nn.Module):
    """Encoder + dropout + one linear head with three outputs (sigmoid applied in the loss / at scoring)."""

    def __init__(self, model_name, pooling):
        super().__init__()
        kw = {"add_pooling_layer": False} if pooling == "first" else {}
        # eager attention: the SDPA memory-efficient backward is non-deterministic on CUDA
        self.encoder = AutoModel.from_pretrained(model_name, attn_implementation="eager", **kw)
        self.pooling = pooling
        self.dropout = torch.nn.Dropout(DROPOUT)
        self.head = torch.nn.Linear(self.encoder.config.hidden_size, len(LABELS))

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state[:, 0] if self.pooling == "first" else out.pooler_output
        return self.head(self.dropout(h))


class CommentDataset(Dataset):
    def __init__(self, enc, y):
        self.enc, self.y = enc, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i],
                "labels": self.y[i]}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_loader(d, spec, tok, shuffle, seed):
    enc = tok(list(prep(d["text"], spec)), truncation=True, max_length=MAX_LEN)
    y = d[LABELS].to_numpy(dtype=np.float32)

    def collate(items):
        labels = torch.tensor(np.stack([it.pop("labels") for it in items]))
        batch = tok.pad(items, return_tensors="pt")
        batch["labels"] = labels
        return batch

    g = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(CommentDataset(enc, y), batch_size=BATCH, shuffle=shuffle, generator=g,
                      collate_fn=collate)


def predict(model, loader):
    """Sigmoid scores (n, 3) and mean BCE over all label cells."""
    model.eval()
    scores, loss_sum, n = [], 0.0, 0
    with torch.no_grad():
        for b in loader:
            y = b["labels"].to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
            logits = logits.float()
            loss_sum += F.binary_cross_entropy_with_logits(logits, y, reduction="sum").item()
            n += y.numel()
            scores.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(scores), loss_sum / n


def split(k):
    val_k = (k + 1) % N_FOLDS
    return df[~df["fold"].isin([k, val_k])], df[df["fold"] == val_k], df[df["fold"] == k]


def train_run(system, seed, k):
    spec, tok = SYSTEMS[system], TOKENIZERS[system]
    tr, va, te = split(k)
    t0 = time.time()
    torch.cuda.reset_peak_memory_stats()
    set_seed(seed)
    model = MultiLabelModel(spec["model"], spec["pooling"]).to(DEVICE)
    tr_dl = make_loader(tr, spec, tok, True, seed)
    va_dl = make_loader(va, spec, tok, False, seed)
    te_dl = make_loader(te, spec, tok, False, seed)

    # no weight decay on biases and LayerNorm weights (1-D parameters)
    params = list(model.named_parameters())
    opt = torch.optim.AdamW([{"params": [p for _, p in params if p.ndim >= 2], "weight_decay": WEIGHT_DECAY},
                             {"params": [p for _, p in params if p.ndim < 2], "weight_decay": 0.0}], lr=LR)
    n_micro = len(tr_dl)
    steps_per_epoch = math.ceil(n_micro / ACCUM)
    total_steps = steps_per_epoch * MAX_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, round(WARMUP * total_steps), total_steps)
    # samples per accumulation group, so each optimizer step uses the exact mean over its effective batch
    sizes = [min(BATCH, len(tr) - j * BATCH) for j in range(n_micro)]
    group_n = [sum(sizes[g * ACCUM:(g + 1) * ACCUM]) for g in range(steps_per_epoch)]

    best_loss, best_state, best_epoch, bad, history = math.inf, None, 0, 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        train_loss = 0.0
        for i, b in enumerate(tr_dl):
            y = b["labels"].to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
            loss_sum = F.binary_cross_entropy_with_logits(logits.float(), y, reduction="sum")
            (loss_sum / (group_n[i // ACCUM] * len(LABELS))).backward()
            train_loss += loss_sum.item()
            if (i + 1) % ACCUM == 0 or i == n_micro - 1:
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
        _, val_loss = predict(model, va_dl)
        history.append({"epoch": epoch, "train_loss": train_loss / (len(tr) * len(LABELS)),
                        "val_loss": val_loss})
        if val_loss < best_loss:
            best_loss, best_epoch, bad = val_loss, epoch, 0
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    va_s, _ = predict(model, va_dl)
    te_s, _ = predict(model, te_dl)

    test = te[["id", "lang", "fold"] + LABELS].copy()
    val = va[["id"] + LABELS].copy()
    choices = []
    for j, lab in enumerate(LABELS):
        val[f"score_{lab}"] = va_s[:, j]
        ytr = tr[lab].to_numpy()
        if len(np.unique(ytr)) < 2:               # single-class training label: constant predictor
            c = int(ytr[0])
            test[f"score_{lab}"], test[f"pred_{lab}"] = float(c), c
            choices.append({"label": lab, "threshold": None, "val_f1": None, "note": "constant predictor"})
            continue
        t, f = best_threshold(va[lab].to_numpy(), va_s[:, j])
        test[f"score_{lab}"] = te_s[:, j]
        test[f"pred_{lab}"] = (te_s[:, j] >= t).astype(int)
        choices.append({"label": lab, "threshold": t, "val_f1": f, "note": ""})

    meta = {"system": system, "seed": seed, "fold": k, "n_train": len(tr), "n_val": len(va),
            "n_test": len(te), "best_epoch": best_epoch, "epochs_run": len(history),
            "best_val_loss": best_loss, "runtime_s": round(time.time() - t0, 1),
            "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
            "history": history, "choices": choices}
    del model, opt, sched, best_state
    gc.collect()
    torch.cuda.empty_cache()
    return test, val, meta


def run_paths(system, seed, k):
    stem = RUNS / f"{system}_seed{seed}_fold{k}"
    return stem.with_name(stem.name + "_test.csv"), stem.with_name(stem.name + "_val.csv"), \
        stem.with_name(stem.name + "_meta.json")

## Run all (system, seed, fold) combinations — cached, safe to interrupt and re-run

In [6]:
jobs = [(s, seed, k) for s in SYSTEMS for seed in SEEDS for k in FOLDS]
t_start = time.time()
for i, (system, seed, k) in enumerate(jobs, 1):
    p_test, p_val, p_meta = run_paths(system, seed, k)
    if p_meta.exists():
        continue
    test, val, meta = train_run(system, seed, k)
    test.to_csv(p_test, index=False)
    val.to_csv(p_val, index=False)
    p_meta.write_text(json.dumps(meta, indent=1))     # written last = run complete
    print(f"[{i:2d}/{len(jobs)}] {system:11s} seed {seed:3d} fold {k}  best epoch {meta['best_epoch']:2d}/"
          f"{meta['epochs_run']:2d}  {meta['runtime_s']:5.1f}s  {meta['peak_gpu_mem_gb']:.2f} GB  "
          f"elapsed {(time.time() - t_start) / 60:.1f} min", flush=True)
missing = [j for j in jobs if not run_paths(*j)[2].exists()]
assert not missing, f"missing runs: {missing}"
print("all runs complete")

[ 1/75] XLM-R       seed  13 fold 0  best epoch  3/ 5    7.8s  5.48 GB  elapsed 0.1 min


[ 2/75] XLM-R       seed  13 fold 1  best epoch  7/ 9   12.5s  5.49 GB  elapsed 0.3 min


[ 3/75] XLM-R       seed  13 fold 2  best epoch  4/ 6    9.0s  5.47 GB  elapsed 0.5 min


[ 4/75] XLM-R       seed  13 fold 3  best epoch  5/ 7   10.0s  5.48 GB  elapsed 0.7 min


[ 5/75] XLM-R       seed  13 fold 4  best epoch  9/10   13.0s  5.49 GB  elapsed 0.9 min


...

[72/75] mBERT-lower seed 100 fold 1  best epoch  7/ 9   10.1s  4.00 GB  elapsed 12.6 min


[73/75] mBERT-lower seed 100 fold 2  best epoch  5/ 7    7.8s  3.99 GB  elapsed 12.7 min


[74/75] mBERT-lower seed 100 fold 3  best epoch  8/10   10.9s  4.00 GB  elapsed 12.9 min


[75/75] mBERT-lower seed 100 fold 4  best epoch  9/10   11.2s  4.00 GB  elapsed 13.1 min


all runs complete


## Aggregate: OOF predictions, per-fold metrics, per-language metrics, summary

In [7]:
PRED = [f"pred_{l}" for l in LABELS]
SCORE = [f"score_{l}" for l in LABELS]
OOF_COLS = ["id", "lang", "fold"] + LABELS + [c for l in LABELS for c in (f"score_{l}", f"pred_{l}")]
ORDER = {i: n for n, i in enumerate(df["id"])}


def in_dataset_order(parts):
    oof = pd.concat(parts)
    return oof.sort_values("id", key=lambda s: s.map(ORDER)).reset_index(drop=True)[OOF_COLS]


def ranking_scores(test):
    out = {}
    for lab in LABELS:
        y, s = test[lab].to_numpy(), test[f"score_{lab}"].to_numpy()
        both = len(np.unique(y)) == 2
        out[f"{lab}_roc_auc"] = roc_auc_score(y, s) if both else np.nan
        out[f"{lab}_avg_precision"] = average_precision_score(y, s) if both else np.nan
    return out


fold_rows, hp_rows, lang_rows, oof_by = [], [], [], {}
for system in SYSTEMS:
    for seed in SEEDS:
        parts = []
        for k in FOLDS:
            p_test, _, p_meta = run_paths(system, seed, k)
            test, meta = pd.read_csv(p_test), json.loads(p_meta.read_text())
            parts.append(test)
            r = evaluate(test[LABELS].to_numpy(), test[PRED].to_numpy())
            r.update(ranking_scores(test))
            r.update({key: meta[key] for key in ("system", "seed", "fold", "n_train", "n_val", "best_epoch",
                                                  "epochs_run", "best_val_loss", "runtime_s", "peak_gpu_mem_gb")})
            fold_rows.append(r)
            hp_rows += [dict(c, system=system, seed=seed, fold=k, best_epoch=meta["best_epoch"])
                        for c in meta["choices"]]
        oof = in_dataset_order(parts)
        oof.to_csv(OUT / f"oof_{system}_seed{seed}.csv", index=False)
        oof_by[(system, seed)] = oof
        lang_rows.append(by_language(oof, PRED).assign(system=system, seed=str(seed)))

per_fold = pd.DataFrame(fold_rows)
per_fold.to_csv(OUT / "per_fold.csv", index=False)
pd.DataFrame(hp_rows).to_csv(OUT / "hyperparameters.csv", index=False)

# seed-mean scores per comment (for the bootstrap): average scores over seeds; per fold and label the
# threshold is chosen with best_threshold on the seed-averaged validation scores, then frozen for the test fold
for system in SYSTEMS:
    parts = []
    for k in FOLDS:
        tests = [pd.read_csv(run_paths(system, s, k)[0]).set_index("id") for s in SEEDS]
        vals = [pd.read_csv(run_paths(system, s, k)[1]).set_index("id") for s in SEEDS]
        test = tests[0][["lang", "fold"] + LABELS].copy()
        test_mean = sum(t[SCORE] for t in tests) / len(tests)
        val_mean = sum(v[SCORE] for v in vals) / len(vals)
        tr, _, _ = split(k)
        for lab in LABELS:
            ytr = tr[lab].to_numpy()
            if len(np.unique(ytr)) < 2:
                test[f"score_{lab}"], test[f"pred_{lab}"] = float(ytr[0]), int(ytr[0])
                continue
            t, _ = best_threshold(vals[0].loc[val_mean.index, lab].to_numpy(), val_mean[f"score_{lab}"].to_numpy())
            test[f"score_{lab}"] = test_mean[f"score_{lab}"]
            test[f"pred_{lab}"] = (test_mean[f"score_{lab}"] >= t).astype(int)
        parts.append(test.reset_index())
    oof = in_dataset_order(parts)
    oof.to_csv(OUT / f"oof_{system}_seedmean.csv", index=False)
    oof_by[(system, "mean")] = oof
    lang_rows.append(by_language(oof, PRED).assign(system=system, seed="mean"))

by_lang = pd.concat(lang_rows)
by_lang.to_csv(OUT / "by_language_oof.csv")

In [8]:
# summary in the format of results/classical/summary.csv:
#   *_mean / *_sd = over test folds of the seed-averaged fold metrics (comparable to the classical SD)
#   macro_f1_seed_sd = SD over seeds of the 5-fold mean macro-F1
#   macro_f1_<lang> = mean over seeds of the per-seed pooled out-of-fold macro-F1
METRICS = ["macro_f1", "micro_f1", "exact_match"] + [f"{l}_f1" for l in LABELS]
RANK = [f"{l}_{m}" for l in LABELS for m in ("roc_auc", "avg_precision")]
summary = []
for system in SYSTEMS:
    pf = per_fold[per_fold["system"] == system]
    fold_means = pf.groupby("fold")[METRICS + RANK].mean()
    row = {"system": system}
    for m in METRICS:
        row[f"{m}_mean"], row[f"{m}_sd"] = fold_means[m].mean(), fold_means[m].std(ddof=1)
    bl = by_lang[(by_lang["system"] == system) & (by_lang["seed"] != "mean")]
    for lang in LANGS:
        row[f"macro_f1_{lang}"] = bl.loc[lang, "macro_f1"].mean()
    row["dCS"] = row["macro_f1_KK"] - row["macro_f1_CS"]
    row["macro_f1_seed_sd"] = pf.groupby("seed")["macro_f1"].mean().std(ddof=1)
    for m in RANK:
        row[f"{m}_mean"] = fold_means[m].mean()
    row["n_seeds"], row["n_runs"] = len(SEEDS), len(pf)
    summary.append(row)
summary = pd.DataFrame(summary)
summary.to_csv(OUT / "summary.csv", index=False)

run_info = {"env": ENV, "model_revisions": REVISIONS, "smoke": SMOKE, "seeds": SEEDS, "folds": FOLDS,
            "n_comments": len(df), "max_len": MAX_LEN, "lr": LR, "weight_decay": WEIGHT_DECAY,
            "weight_decay_excludes": "biases and LayerNorm weights (1-D parameters)",
            "micro_batch": BATCH, "grad_accum": ACCUM, "effective_batch": BATCH * ACCUM,
            "warmup_share": WARMUP, "scheduler": "linear decay after warm-up", "max_epochs": MAX_EPOCHS,
            "early_stopping": f"validation BCE, patience {PATIENCE}, best checkpoint restored",
            "dropout_head": DROPOUT, "precision": "bf16 autocast", "optimizer": "torch AdamW",
            "attention": "eager (deterministic)",
            "pooling": {s: v["pooling"] for s, v in SYSTEMS.items()},
            "protocol": "5-fold grouped CV; validation = next fold; threshold frozen before test",
            "total_train_runtime_s": round(float(per_fold["runtime_s"].sum()), 1),
            "max_peak_gpu_mem_gb": float(per_fold["peak_gpu_mem_gb"].max())}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(f"wrote {OUT}/  (total training time {run_info['total_train_runtime_s'] / 60:.1f} min)")

wrote results\transformers/  (total training time 12.9 min)


## Checks and comparison with the classical baselines

In [9]:
classical = pd.read_csv("results/classical/summary.csv")
for key, oof in oof_by.items():
    n_expected = int(df["fold"].isin(FOLDS).sum())
    assert len(oof) == n_expected and oof["id"].is_unique, key
    assert not oof[PRED + SCORE].isna().any().any(), key
# the all-positive reference recomputed from the same folds must equal the classical value
ref = np.mean([evaluate(df[df.fold == k][LABELS].to_numpy(), np.ones((int((df.fold == k).sum()), 3)))["macro_f1"]
               for k in range(N_FOLDS)])
ref_classical = classical.loc[classical.system == "All-positive (reference)", "macro_f1_mean"].item()
assert abs(ref - ref_classical) < 1e-9, (ref, ref_classical)
print(f"checks passed: OOF files complete, ids unique, no missing predictions, reference {ref:.3f} reproduced")

show = pd.concat([classical[classical.system.isin(["All-positive (reference)", "LR-TFIDF"])], summary])
cols = ["system", "macro_f1_mean", "macro_f1_sd", "macro_f1_seed_sd", "toxic_f1_mean", "offensive_f1_mean",
        "hate_f1_mean", "macro_f1_KK", "macro_f1_RU", "macro_f1_CS", "dCS"]
print(show[cols].round(3).to_string(index=False))
print("\nmedian best epoch:", per_fold.groupby("system")["best_epoch"].median().to_dict())

checks passed: OOF files complete, ids unique, no missing predictions, reference 0.331 reproduced
                  system  macro_f1_mean  macro_f1_sd  macro_f1_seed_sd  toxic_f1_mean  offensive_f1_mean  hate_f1_mean  macro_f1_KK  macro_f1_RU  macro_f1_CS    dCS
All-positive (reference)          0.331        0.020               NaN          0.496              0.380         0.117        0.270        0.387        0.346 -0.076
                LR-TFIDF          0.319        0.083               NaN          0.484              0.304         0.168        0.233        0.394        0.315 -0.081
                   XLM-R          0.298        0.031             0.023          0.442              0.375         0.077        0.260        0.361        0.290 -0.030
                   mBERT          0.338        0.028             0.037          0.509              0.406         0.100        0.306        0.403        0.308 -0.002
             mBERT-lower          0.331        0.024             0.020       